## Configuration and Imports

In [ ]:
from datasets import load_dataset
from tqdm.notebook import tqdm
import json
import os

CODE_KEYWORDS = {
    "def ",
    "class ",
    "import ",
    "return ",
    "print(",
    "function",
    "python",
    "javascript",
    "html",
    "css",
    "c++",
    "java",
    "sql",
    "compile",
    "debug",
    "variable",
    "array",
    "syntax",
    "algorithm",
    "```",
}

MATH_KEYWORDS = {
    "calculate",
    "equation",
    "theorem",
    "integral",
    "derivative",
    "algebra",
    "geometry",
    "math",
    "\\frac",
    "\\sqrt",
    "sum of",
}

MED_KEYWORDS = {
    "symptom",
    "diagnosis",
    "treatment",
    "patient",
    "syndrome",
    "mg ",
    "dosage",
    "clinical",
    "doctor",
    "disease",
    "infection",
}

output_path_ultrachat = "../data/filtered/clean_ultrachat.json"
output_path_squad = "../data/filtered/clean_squad.json"


## Apply Filter

### Ultrachat

In [ ]:
# 1. Extractor function for UltraChat
def ultrachat_extractor(row):
    messages = row.get("messages", [])
    if messages and len(messages) > 0 and messages[0]["role"] == "user":
        return messages[0]["content"]
    return None


print("Loading dataset to determine total size...")
# Loading normally (without streaming) to get the exact length
ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")

# 2. Calculate the 90% cutoff dynamically
total_rows = len(ds)
target_scan_amount = int(total_rows * 0.90)

print(f"Total rows in dataset: {total_rows:,}")
print(f"Scanning 90% of dataset: {target_scan_amount:,} rows\n")

clean_prompts = []
dropped_count = 0

# 3. Process the rows up to the 90% mark
for i in tqdm(range(target_scan_amount), desc="Filtering Prompts"):
    row = ds[i]
    text = ultrachat_extractor(row)

    if not text:
        continue

    text_lower = text.lower()

    # Check against filter
    is_dirty = (
        any(kw in text_lower for kw in CODE_KEYWORDS)
        or any(kw in text_lower for kw in MATH_KEYWORDS)
        or any(kw in text_lower for kw in MED_KEYWORDS)
    )

    if is_dirty:
        dropped_count += 1
    else:
        clean_prompts.append(text)

# 4. Save to JSON array format


with open(output_path_ultrachat, "w", encoding="utf-8") as f:
    # This dumps the data exactly in the ["prompt 1", "prompt 2"] format you shared
    json.dump(clean_prompts, f, indent=4, ensure_ascii=False)

print("\n--- Filtering Complete ---")
print(f"Total Rows Scanned: {target_scan_amount:,}")
print(
    f"Clean Collected:    {len(clean_prompts):,} ({(len(clean_prompts) / target_scan_amount) * 100:.1f}%)"
)
print(
    f"Dirty Dropped:      {dropped_count:,} ({(dropped_count / target_scan_amount) * 100:.1f}%)"
)
print(f"Saved cleanly to:   {output_path_ultrachat}")


### Squad

In [ ]:
# 1. Extractor function specifically for SQuAD v2
def squad_extractor(row):
    # SQuAD prompts are stored in the 'question' key
    return row.get("question", None)


print("Loading SQuAD v2 dataset to determine total size...")
ds_squad = load_dataset("squad_v2", split="train")

# 2. Calculate the 90% cutoff dynamically
total_rows_squad = len(ds_squad)
target_scan_amount_squad = int(total_rows_squad * 0.90)

print(f"Total rows in SQuAD dataset: {total_rows_squad:,}")
print(f"Scanning 90% of dataset: {target_scan_amount_squad:,} rows\n")

clean_squad_prompts = []
dropped_squad_count = 0

# 3. Process the rows up to the 90% mark
for i in tqdm(range(target_scan_amount_squad), desc="Filtering SQuAD Prompts"):
    row = ds_squad[i]
    text = squad_extractor(row)

    if not text:
        continue

    text_lower = text.lower()

    # Check against our blacklists (reusing the lists from Cell 1)
    is_dirty = (
        any(kw in text_lower for kw in CODE_KEYWORDS)
        or any(kw in text_lower for kw in MATH_KEYWORDS)
        or any(kw in text_lower for kw in MED_KEYWORDS)
    )

    if is_dirty:
        dropped_squad_count += 1
    else:
        clean_squad_prompts.append(text)

# 4. Save to JSON array format


with open(output_path_squad, "w", encoding="utf-8") as f:
    json.dump(clean_squad_prompts, f, indent=4, ensure_ascii=False)

print("\n--- SQuAD Filtering Complete ---")
print(f"Total Rows Scanned: {target_scan_amount_squad:,}")
print(
    f"Clean Collected:    {len(clean_squad_prompts):,} ({(len(clean_squad_prompts) / target_scan_amount_squad) * 100:.1f}%)"
)
print(
    f"Dirty Dropped:      {dropped_squad_count:,} ({(dropped_squad_count / target_scan_amount_squad) * 100:.1f}%)"
)
print(f"Saved cleanly to:   {output_path_squad}")
